# DeepGlobe Satellite Segmentation Launcher

This is the Master notebook designed to launch the project on Colab utilizing the T4 GPU.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, shutil

print('=== 1. CLONING GITHUB REPOSITORY ===')
REPO_URL = 'https://github.com/robertopassante/Neural-Networks-Project.git'
REPO_DIR = '/content/Neural-Networks-Project'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin main

%cd {REPO_DIR}

print('\n=== 2. INSTALLING DEPENDENCIES ===')
!pip install -q transformers minigpt4 tqdm matplotlib
print('Done.')

print('\n=== 3. COPYING DATASET TO VIRTUAL SSD ===')
ZIP_PATH = '/content/drive/MyDrive/satellite_segmentation/DeepGlobe Land Cover Classification Dataset-001.zip'
LOCAL_DATA = '/content/dataset_local'

if not os.path.exists(LOCAL_DATA) or len(os.listdir(LOCAL_DATA)) == 0:
    import zipfile
    from tqdm.auto import tqdm
    os.makedirs(LOCAL_DATA, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        for member in tqdm(z.infolist(), desc='Extracting', unit='file'):
            z.extract(member, LOCAL_DATA)
    print('Extraction complete!')
else:
    print('Dataset already on local SSD.')

TRAIN_DIR = None
for root, dirs, files in os.walk(LOCAL_DATA):
    if 'train' in dirs:
        TRAIN_DIR = os.path.join(root, 'train')
        break
print(f'Train folder: {TRAIN_DIR}')


Current working directory: /content/drive/MyDrive/satellite_segmentation


In [3]:
import sys
import torch

# Assicuriamo che il notebook legga i moduli dal repo appena clonato
sys.path.insert(0, REPO_DIR)

from data.dataset_manager import get_dataloader
from models.satmae_unet import SatMAESegmentor
from training.trainer import SatMAETrainer

# === CONFIGURAZIONE ===
BATCH_SIZE = 16
NUM_EPOCHS = 30
LR = 1e-4
CHECKPOINT_DIR = os.path.join(REPO_DIR, 'checkpoints')

# === AVVIO ===
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')

train_loader = get_dataloader(TRAIN_DIR, batch_size=BATCH_SIZE)
model = SatMAESegmentor(num_classes=7, in_channels=3).to(device)

trainer = SatMAETrainer(
    model=model,
    train_loader=train_loader,
    device=device,
    lr=LR,
    checkpoint_dir=CHECKPOINT_DIR
    # NOTE: drive_checkpoint_dir è già preconfigurato su Drive dentro trainer.py per i salvataggi automatici!
)

trainer.train(num_epochs=NUM_EPOCHS)

# Push finale (Opzionale, pusha i grafici finali su Google drive)
print('\n=== BACKUP RISULTATI SU DRIVE ===')
!cp -r {REPO_DIR}/training_results /content/drive/MyDrive/satellite_segmentation/ 2>/dev/null || true


Installing dependencies...
